In [2]:
from pathlib import Path

txt_path = Path("../data/MACCROBAT2018/15939911.txt")

text = txt_path.read_text(encoding="utf-8")

print(text)

CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.
The symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.
Except for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.
An electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.
Transthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).
The anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal leaflet was rudimentary (Fig.2C, arrowhead)

In [3]:
ann_path = Path("../data/MACCROBAT2018/15939911.ann")

annotations =ann_path.read_text(encoding="utf-8")

print(annotations[:2000])

T1	Age 8 19	28-year-old
T2	History 20 38	previously healthy
T3	Sex 39 42	man
T4	Clinical_event 43 52	presented
E1	Clinical_event:T4 
T5	Sign_symptom 31 38	healthy
E2	Sign_symptom:T5 
T6	Duration 60 66	6-week
E3	Duration:T6 
T7	Sign_symptom 78 90	palpitations
E4	Sign_symptom:T7 
T8	Coreference 96 104	symptoms
E5	Coreference:T8 
R1	IDENTICAL Arg1:E5 Arg2:E4	
T9	Clinical_event 121 125	rest
E6	Clinical_event:T9 
R2	MODIFY Arg1:E6 Arg2:E5	
T10	Frequency 127 145	2–3 times per week
R3	MODIFY Arg1:T10 Arg2:E5	
T12	Sign_symptom 206 213	dyspnea
E8	Sign_symptom:T12 
T11	Detailed_description 154 180	up to 30 minutes at a time
R4	MODIFY Arg1:T11 Arg2:E5	
T13	Sign_symptom 261 281	regurgitation murmur
E7	Sign_symptom:T13 
T14	Biological_structure 251 260	tricuspid
T15	Detailed_description 238 250	holosystolic
T16	Lab_value 228 237	grade 2/6
R5	MODIFY Arg1:T14 Arg2:E7	
R6	MODIFY Arg1:T15 Arg2:E7	
R7	MODIFY Arg1:T16 Arg2:E7	
T17	Biological_structure 301 320	left sternal border
R8	MODIFY Arg1:T17 Arg2:E

In [4]:
entity_lines = []

for line in annotations.splitlines():
    if line.startswith("T"):
        entity_lines.append(line)

print("Number of entities:", len(entity_lines))
print("\nFirst 10 entites")

for line in entity_lines[:10]:
    print(line)

Number of entities: 68

First 10 entites
T1	Age 8 19	28-year-old
T2	History 20 38	previously healthy
T3	Sex 39 42	man
T4	Clinical_event 43 52	presented
T5	Sign_symptom 31 38	healthy
T6	Duration 60 66	6-week
T7	Sign_symptom 78 90	palpitations
T8	Coreference 96 104	symptoms
T9	Clinical_event 121 125	rest
T10	Frequency 127 145	2–3 times per week


In [5]:
import pandas as pd

entities = []

for line in entity_lines:
    parts = line.split("\t")

    entity_id = parts[0]
    entity_info = parts[1].split()
    entity_text = parts[2]

    entity_type = entity_info[0]
    start = int(entity_info[1])
    end = int(entity_info[2])

    entities.append({
        "entity_id" : entity_id,
        "type" : entity_type,
        "start" : start,
        "end" : end,
        "text" : entity_text
    })

df_entities = pd.DataFrame(entities)

df_entities.head(20)

,entity_id,type,start,end,text
0,T1,Age,8,19,28-year-old
1,T2,History,20,38,previously healthy
2,T3,Sex,39,42,man
3,T4,Clinical_event,43,52,presented
4,T5,Sign_symptom,31,38,healthy
5,T6,Duration,60,66,6-week
6,T7,Sign_symptom,78,90,palpitations
7,T8,Coreference,96,104,symptoms
8,T9,Clinical_event,121,125,rest
9,T10,Frequency,127,145,2–3 times per week


In [6]:
data_dir = Path("../data/MACCROBAT2018")

txt_files = sorted(data_dir.glob("*.txt"))
ann_files = sorted(data_dir.glob("*.ann"))

print("Text files:", len(txt_files))
print("Annotation files:", len(ann_files))

Text files: 200
Annotation files: 200


In [7]:
txt_ids = {file.stem for file in txt_files}
ann_ids = {file.stem for file in ann_files}

print("TXT Ids:", len(txt_ids))
print("ANN Ids:", len(ann_ids))

print("TXT without ANN:", txt_ids - ann_ids)
print("ANN without TXT:", ann_ids - txt_ids)

TXT Ids: 200
ANN Ids: 200
TXT without ANN: set()
ANN without TXT: set()


In [8]:
def parse_ann_file(ann_path):
    entities = []

    annotations = ann_path.read_text(encoding="utf-8")

    for line in annotations.splitlines():
        if line.startswith("T"):
            parts = line.split("\t", 2)

            entity_id = parts[0]
            entity_info = parts[1]
            entity_text = parts[2]

            first_space = entity_info.find(" ")
            entity_type = entity_info[:first_space]
            span_info = entity_info[first_space + 1:]

            spans = []

            for span in span_info.split(";"):
                start, end = span.split()
                spans.append((int(start), int(end)))

            entities.append({
                "entity_id": entity_id,
                "entity_type": entity_type,
                "spans": spans,
                "entity_text": entity_text
            })

    return entities

In [9]:
sample_entities = parse_ann_file(Path("../data/MACCROBAT2018/15939911.ann"))

sample_entities[:5]

[{'entity_id': 'T1',
  'entity_type': 'Age',
  'spans': [(8, 19)],
  'entity_text': '28-year-old'},
 {'entity_id': 'T2',
  'entity_type': 'History',
  'spans': [(20, 38)],
  'entity_text': 'previously healthy'},
 {'entity_id': 'T3',
  'entity_type': 'Sex',
  'spans': [(39, 42)],
  'entity_text': 'man'},
 {'entity_id': 'T4',
  'entity_type': 'Clinical_event',
  'spans': [(43, 52)],
  'entity_text': 'presented'},
 {'entity_id': 'T5',
  'entity_type': 'Sign_symptom',
  'spans': [(31, 38)],
  'entity_text': 'healthy'}]

In [10]:
all_entities = []

for txt_path in txt_files:
    document_id = txt_path.stem

    ann_path = data_dir/ f"{document_id}.ann"

    entities = parse_ann_file(ann_path)

    for entity in entities:
        entity["document_id"] = document_id
        all_entities.append(entity)

df_all_entities = pd.DataFrame(all_entities)

print("Total entities:", len(df_all_entities))

df_all_entities.head()


Total entities: 25041


,entity_id,entity_type,spans,entity_text,document_id
0,T1,Age,"[(8, 19)]",28-year-old,15939911
1,T2,History,"[(20, 38)]",previously healthy,15939911
2,T3,Sex,"[(39, 42)]",man,15939911
3,T4,Clinical_event,"[(43, 52)]",presented,15939911
4,T5,Sign_symptom,"[(31, 38)]",healthy,15939911


In [11]:
df_all_entities["entity_type"].value_counts()

entity_type
Diagnostic_procedure      4567
Sign_symptom              3359
Biological_structure      2931
Detailed_description      2901
Lab_value                 2858
Disease_disorder          1362
Medication                1076
Therapeutic_procedure     1005
Date                       731
Clinical_event             626
History                    392
Severity                   369
Dosage                     362
Nonbiological_location     354
Coreference                313
Duration                   280
Age                        206
Sex                        191
Administration             175
Distance                   122
Activity                   108
Family_history              81
Frequency                   76
Shape                       65
Time                        57
Personal_background         57
Subject                     54
Color                       52
Texture                     46
Area                        43
Outcome                     42
Qualitative_concept        

In [12]:
print("Total_entities:", len(df_all_entities))
print("Entity_types:", df_all_entities["entity_type"].nunique())
print("Documents:", df_all_entities["document_id"].nunique())


Total_entities: 25041
Entity_types: 41
Documents: 200


In [13]:
documents = []

for txt_path in txt_files:
    document_id = txt_path.stem
    text = txt_path.read_text(encoding='utf-8')
    documents.append({
        "document_id": document_id,
        "text": text
    })

df_documents = pd.DataFrame(documents)

print("Total documents:", len(df_documents))
print(df_documents.head(5))

Total documents: 200
  document_id                                               text
0    15939911  CASE: A 28-year-old previously healthy man pre...
1    16778410  The patient was a 34-yr-old man who presented ...
2    17803823  A 23 year old white male with a 4 year history...
3    18236639  A 30-year-old female (65 kg) underwent rhinopl...
4    18258107  Here, we describe another case in a 60-year-ol...


In [14]:
sample_id = "15939911"

sample_text = df_documents[df_documents["document_id"] == sample_id]["text"].iloc[0]

sample_entities = df_all_entities[df_all_entities["document_id"] == sample_id]

sample_entities.head(10)

,entity_id,entity_type,spans,entity_text,document_id
0,T1,Age,"[(8, 19)]",28-year-old,15939911
1,T2,History,"[(20, 38)]",previously healthy,15939911
2,T3,Sex,"[(39, 42)]",man,15939911
3,T4,Clinical_event,"[(43, 52)]",presented,15939911
4,T5,Sign_symptom,"[(31, 38)]",healthy,15939911
5,T6,Duration,"[(60, 66)]",6-week,15939911
6,T7,Sign_symptom,"[(78, 90)]",palpitations,15939911
7,T8,Coreference,"[(96, 104)]",symptoms,15939911
8,T9,Clinical_event,"[(121, 125)]",rest,15939911
9,T10,Frequency,"[(127, 145)]",2–3 times per week,15939911


In [15]:
for _, row in sample_entities.iterrows():
    for start, end in row["spans"]:
        extracted_text = sample_text[start:end]

        print("Entity type:", row["entity_type"])
        print("Annotated text:", row["entity_text"])
        print("Text from span:", extracted_text)
        print("-" * 40) 

Entity type: Age
Annotated text: 28-year-old
Text from span: 28-year-old
----------------------------------------
Entity type: History
Annotated text: previously healthy
Text from span: previously healthy
----------------------------------------
Entity type: Sex
Annotated text: man
Text from span: man
----------------------------------------
Entity type: Clinical_event
Annotated text: presented
Text from span: presented
----------------------------------------
Entity type: Sign_symptom
Annotated text: healthy
Text from span: healthy
----------------------------------------
Entity type: Duration
Annotated text: 6-week
Text from span: 6-week
----------------------------------------
Entity type: Sign_symptom
Annotated text: palpitations
Text from span: palpitations
----------------------------------------
Entity type: Coreference
Annotated text: symptoms
Text from span: symptoms
----------------------------------------
Entity type: Clinical_event
Annotated text: rest
Text from span: rest


In [16]:
discontinuous_entities = df_all_entities[df_all_entities["spans"].apply(len)>1]

print("Discontinuous entities:", len(discontinuous_entities))

discontinuous_entities.head()

Discontinuous entities: 55


,entity_id,entity_type,spans,entity_text,document_id
323,T79,Detailed_description,"[(1366, 1373), (1386, 1404)]",without rebound tenderness,17803823
544,T18,Diagnostic_procedure,"[(341, 350), (357, 358)]",Hepatitis C,18416479
1300,T121,Biological_structure,"[(2001, 2005), (2034, 2037)]",left arm,19214295
1303,T124,Diagnostic_procedure,"[(2114, 2133), (2139, 2143)]",computed tomography scan,19214295
1332,T154,Biological_structure,"[(2428, 2438), (2444, 2463)]",origins of subclavian arteries,19214295


In [17]:
document_ids = df_documents["document_id"].tolist()

print("Number of document IDs:", len(document_ids))
print(document_ids[:5])

Number of document IDs: 200
['15939911', '16778410', '17803823', '18236639', '18258107']


In [18]:
from sklearn.model_selection import train_test_split

train_ids, temp_ids = train_test_split(document_ids, test_size=0.3, random_state=42)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

print("Train documents:", len(train_ids))
print("Validation documents:", len(val_ids))
print("Test documents:", len(test_ids))

Train documents: 140
Validation documents: 30
Test documents: 30


In [19]:
train_documents = df_documents[
    df_documents["document_id"].isin(train_ids)
].copy()

val_documents = df_documents[
    df_documents["document_id"].isin(val_ids)
].copy()

test_documents = df_documents[
    df_documents["document_id"].isin(test_ids)
].copy()

print("Train:", len(train_documents))
print("Validation:", len(val_documents))
print("Test:", len(test_documents))

Train: 140
Validation: 30
Test: 30


In [20]:
train_entities = df_all_entities[
    df_all_entities["document_id"].isin(train_ids)
].copy()

val_entities = df_all_entities[
    df_all_entities["document_id"].isin(val_ids)
].copy()

test_entities = df_all_entities[
    df_all_entities["document_id"].isin(test_ids)
].copy()

print("Train entities:", len(train_entities))
print("Validation entities:", len(val_entities))
print("Test entities:", len(test_entities))

Train entities: 17363
Validation entities: 3997
Test entities: 3681


In [21]:
print("Train vs Validation overlap:",
      set(train_ids) & set(val_ids))

print("Train vs Test overlap:",
      set(train_ids) & set(test_ids))

print("Validation vs Test overlap:",
      set(val_ids) & set(test_ids))

Train vs Validation overlap: set()
Train vs Test overlap: set()
Validation vs Test overlap: set()


## BIO Tag Preparation

In [ ]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.
A 0 1
31 1 4
- 2 5
year 3 9
- 4 10
old 5 13
man 6 17
developed 7 27
diabetes 8 36
insipidus 9 46
with 10 51
urine 11 57
volume 12 64
up 13 67
to 14 70
10 15 73
to 16 76
20 17 79
  18 80
L 19 81


In [24]:
import spacy

nlp = spacy.blank("en")

sample_id = train_ids[0]

sample_text = train_documents[train_documents["document_id"]==sample_id]["text"].iloc[0]

doc = nlp(sample_text)

for token in list(doc)[:20]:
    print(token.text, token.idx, token.idx+len(token.text))

A 0 1
31 2 4
- 4 5
year 5 9
- 9 10
old 10 13
man 14 17
developed 18 27
diabetes 28 36
insipidus 37 46
with 47 51
urine 52 57
volume 58 64
up 65 67
to 68 70
10 71 73
to 74 76
20 77 79
  79 80
L 80 81


In [33]:
bio_labels = []

for token in doc:
    label = "O"

    token_start = token.idx
    token_end = token.idx + len(token.text)

    for _, row in sample_entities.iterrows():
        entity_type = row["entity_type"]

        for start, end in row["spans"]:
            if token_start >= start and token_end <= end:
                if token_start == start:
                    label = f"B-{entity_type}"
                else:
                    label = f"I-{entity_type}"
                break

        if label != "O":
            break

    bio_labels.append(label)

In [34]:
for token, label in list(zip(doc, bio_labels))[:30]:
    print(f"{token.text:<15} {label}")

A               O
31              B-Age
-               I-Age
year            I-Age
-               I-Age
old             I-Age
man             B-Sex
developed       O
diabetes        B-Disease_disorder
insipidus       I-Disease_disorder
with            O
urine           B-Diagnostic_procedure
volume          I-Diagnostic_procedure
up              O
to              O
10              B-Volume
to              I-Volume
20              I-Volume
                I-Volume
L               I-Volume
every           B-Frequency
24              I-Frequency
                I-Frequency
hours           I-Frequency
in              O
2003            B-Date
.               O

               O
Four            B-Date
years           I-Date


In [31]:
sample_id = train_ids[0]

sample_text = train_documents[
    train_documents["document_id"] == sample_id
]["text"].iloc[0]

sample_entities = train_entities[
    train_entities["document_id"] == sample_id
].copy()

print("Sample document:", sample_id)
print("Entities:", len(sample_entities))

Sample document: 28248858
Entities: 116


In [32]:
doc = nlp(sample_text)

In [35]:
def create_bio_labels(text, entities):
    doc = nlp(text)
    bio_labels = []

    for token in doc:
        label = "O"

        token_start = token.idx
        token_end = token.idx + len(token.text)

        for _, row in entities.iterrows():
            entity_type = row["entity_type"]

            for start, end in row["spans"]:
                if token_start >= start and token_end <= end:
                    if token_start == start:
                        label = f"B-{entity_type}"
                    else:
                        label = f"I-{entity_type}"
                    break

            if label != "O":
                break

        bio_labels.append(label)

    return doc, bio_labels

In [36]:
doc, bio_labels = create_bio_labels(
    sample_text,
    sample_entities
)

In [37]:
for token, label in list(zip(doc, bio_labels))[:30]:
    print(f"{token.text:<15} {label}")

A               O
31              B-Age
-               I-Age
year            I-Age
-               I-Age
old             I-Age
man             B-Sex
developed       O
diabetes        B-Disease_disorder
insipidus       I-Disease_disorder
with            O
urine           B-Diagnostic_procedure
volume          I-Diagnostic_procedure
up              O
to              O
10              B-Volume
to              I-Volume
20              I-Volume
                I-Volume
L               I-Volume
every           B-Frequency
24              I-Frequency
                I-Frequency
hours           I-Frequency
in              O
2003            B-Date
.               O

               O
Four            B-Date
years           I-Date


In [40]:
def create_bio_labels(text, entities):
    doc = nlp(text)
    bio_labels = []

    entity_spans = []

    for _, row in entities.iterrows():
        entity_type = row["entity_type"]

        for start, end in row["spans"]:
            entity_spans.append((start, end, entity_type))

    entity_spans.sort()

    for token in doc:
        label = "O"

        token_start = token.idx
        token_end = token.idx + len(token.text)

        for start, end, entity_type in entity_spans:

            if start > token_end:
                break

            if token_start >= start and token_end <= end:
                if token_start == start:
                    label = f"B-{entity_type}"
                else:
                    label = f"I-{entity_type}"
                break

        bio_labels.append(label)

    return doc, bio_labels

In [41]:
import time

start_time = time.time()

doc, bio_labels = create_bio_labels(
    sample_text,
    sample_entities
)

end_time = time.time()

print("Time:", end_time - start_time, "seconds")

Time: 0.025945425033569336 seconds


In [42]:
test_ner_data = []

for _, row in train_documents.head(10).iterrows():
    document_id = row["document_id"]
    text = row["text"]

    entities = train_entities[
        train_entities["document_id"] == document_id
    ]

    doc, bio_labels = create_bio_labels(text, entities)

    test_ner_data.append({
        "document_id": document_id,
        "tokens": [token.text for token in doc],
        "labels": bio_labels
    })

In [43]:
print("Documents processed:", len(test_ner_data))

Documents processed: 10


In [44]:
for item in test_ner_data:
    print(
        item["document_id"],
        len(item["tokens"]),
        len(item["labels"])
    )

15939911 323 323
16778410 473 473
17803823 494 494
18236639 371 371
18258107 518 518
18561524 668 668
18666334 231 231
18787726 364 364
19009665 404 404
19214295 567 567


In [45]:
train_ner_data = []

for _, row in train_documents.iterrows():
    document_id = row["document_id"]
    text = row["text"]

    entities = train_entities[
        train_entities["document_id"] == document_id
    ]

    doc, bio_labels = create_bio_labels(text, entities)

    train_ner_data.append({
        "document_id": document_id,
        "tokens": [token.text for token in doc],
        "labels": bio_labels
    })

print("Training documents processed:", len(train_ner_data))

Training documents processed: 140


In [46]:
all_lengths_match = all(
    len(item["tokens"]) == len(item["labels"])
    for item in train_ner_data
)

print("All token/label lengths match:", all_lengths_match)

All token/label lengths match: True


In [ ]:
def prepare_ner_data(documents, entities):
    ner_data = []

    for _, row in documents.iterrows():
        document_id = row["document_id"]
        text = row["text"]

        document_entities = entities[
            entities["document_id"] == document_id
        ]

        doc, bio_labels = create_bio_labels(
            text,
            document_entities
        )

        ner_data.append({
            "document_id": document_id,
            "tokens": [token.text for token in doc],
            "labels": bio_labels
        })

    return ner_data

In [48]:
val_ner_data = prepare_ner_data(
    val_documents,
    val_entities
)

test_ner_data = prepare_ner_data(
    test_documents,
    test_entities
)

print("Train:", len(train_ner_data))
print("Validation:", len(val_ner_data))
print("Test:", len(test_ner_data))

Train: 140
Validation: 30
Test: 30


In [49]:
train_types = set(train_entities["entity_type"].unique())
val_types = set(val_entities["entity_type"].unique())
test_types = set(test_entities["entity_type"].unique())

print("Train entity types:", len(train_types))
print("Validation entity types:", len(val_types))
print("Test entity types:", len(test_types))

Train entity types: 41
Validation entity types: 40
Test entity types: 38


In [50]:
print("Missing in validation:")
print(train_types - val_types)

print("\nMissing in test:")
print(train_types - test_types)

Missing in validation:
{'Mass'}

Missing in test:
{'Weight', 'Mass', 'Biological_attribute'}


In [51]:
entity_distribution = (
    pd.DataFrame({
        "Train": train_entities["entity_type"].value_counts(),
        "Validation": val_entities["entity_type"].value_counts(),
        "Test": test_entities["entity_type"].value_counts()
    })
    .fillna(0)
    .astype(int)
)

entity_distribution

,Train,Validation,Test
entity_type,,,
Activity,77,22,9
Administration,135,11,29
Age,143,32,31
Area,21,8,14
Biological_attribute,7,3,0
Biological_structure,1948,572,411
Clinical_event,448,79,99
Color,43,8,1
Coreference,242,29,42


In [55]:
import json

with open("../data/train_ner_data.json", "w", encoding="utf-8") as f:
    json.dump(train_ner_data, f, ensure_ascii=False, indent=2)

with open("../data/val_ner_data.json", "w", encoding="utf-8") as f:
    json.dump(val_ner_data, f, ensure_ascii=False, indent=2)

with open("../data/test_ner_data.json", "w", encoding="utf-8") as f:
    json.dump(test_ner_data, f, ensure_ascii=False, indent=2)

In [56]:
print(len(train_ner_data))
print(len(val_ner_data))
print(len(test_ner_data))

140
30
30
